In [ ]:
import numpy as np
import scipy.io.wavfile as scipy_wav
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Audio
from scipy.signal import butter, sosfilt
from scipy.stats import linregress
from dataclasses import dataclass
import pandas as pd
import math
sns.set_theme()

In [ ]:
def wide_plot():
    return plt.subplots(figsize=(12, 6))

In [ ]:
sample_rate, stereo_sound_data \
    = scipy_wav.read("invrev-beeps.wav")

In [ ]:
beep_vals = stereo_sound_data[:, 0] / 32768.0
beep_tms = np.arange(len(beep_vals)) / sample_rate

In [ ]:
plt.plot(beep_tms, beep_vals)

In [ ]:
sos = butter(10, 5, 'hp', fs=sample_rate, output='sos')
beep_hp = sosfilt(sos, beep_vals)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
#idxs = slice(round(6.23*sample_rate), round(6.53*sample_rate))
idxs = slice(round(12.9*sample_rate), round(13.9*sample_rate))
#idxs = slice(round(26.1*sample_rate), round(27.0*sample_rate))
ax.plot(beep_tms[idxs], beep_vals[idxs])

In [ ]:
FS_OUT = 48000


@dataclass
class Chirp:
    period_0: float
    d_period: float
    duration: float

    def synthesised(self, wave_duration):
        xp1 = 0.0
        yp1 = 0.0
        xp = [xp1]
        yp = [yp1]
        period = self.period_0
        while xp[-1] < wave_duration or (len(xp) % 2 == 0):
            xp1 += period
            yp1 += math.pi
            xp.append(xp1)
            yp.append(yp1)
            period += self.d_period
        
        node_ts = np.array(xp)
        node_phs = np.array(yp)

        full_duration = xp1
        n_samples = int(full_duration * FS_OUT)
        chirp_xs = np.arange(n_samples, dtype=np.uint32)
        chirp_ts = chirp_xs / FS_OUT
        chirp_phs = np.interp(chirp_ts, node_ts, node_phs)
        chirp_ys = np.tanh(6.5 * np.sin(chirp_phs))

        return chirp_ys


@dataclass
class SoundEffect:
    chirps: [Chirp]

    def synthesised(self, chirp_wave_duration):
        return np.concat(
            [ch.synthesised(chirp_wave_duration)
             for ch in self.chirps]
        )

In [ ]:
data = beep_vals[idxs]
n_chirps = 48
fs = sample_rate
n_gaps = n_chirps - 1

crossings = np.where(np.diff((data > 0.5).astype(np.int8)))[0]
# crossings[i] is an index into data such that
#
#   data[crossings[i]] > 0.5  whereas  data[crossings[i] + 1] <= 0.5
#
# or vice versa; in fact cases which change from >0.5 to <=0.5
# alternate with cases which changes from <=0.5 to >0.5.

In [ ]:
fig, ax = wide_plot()
ax.plot(data[3000:45000])
Audio(data, rate=sample_rate)

In [ ]:
half_periods = np.diff(crossings)

# The longest n_breaks "half periods" are in fact gaps between chirps.
s_hperiods = sorted(half_periods, reverse=True)

# s_hperiods[n_gaps-1] should be the shortest gap between chirps,
# and s_hperiods[n_gaps] the longest true half-period of a chirp.
threshold = np.mean([s_hperiods[n_gaps - 1], s_hperiods[n_gaps]])

gap_idxs = [int(i) for i in np.where(half_periods > threshold)[0]]
# Gap (i), zero-based, is between data-indexes crossings[gap_idx[i]]
# and crossings[gap_idx[i] + 1]
pd.DataFrame.from_records(
    ((gi, crossings[gi], crossings[gi+1]) for i, gi in enumerate(gap_idxs)),
    columns=["gap_idx", "data_idx_0", "data_idx_1"]
)
None

In [ ]:
records = []
chirps = []
for gi0, gi1 in zip([0] + gap_idxs, gap_idxs + [len(half_periods)]):
    # Discard last half-period because it might have been truncated
    chirp_hperiods = half_periods[gi0 + 1:gi1 - 1]
    lr_data = linregress(np.arange(len(chirp_hperiods)), chirp_hperiods)
    data_idx_0 = crossings[gi0 + 1]
    data_idx_1 = crossings[gi1] + 1
    chirp_len = data_idx_1 - data_idx_0
    records.append((gi0, gi1, data_idx_0, data_idx_1, chirp_len, lr_data.intercept, lr_data.slope, lr_data.rvalue))
    chirps.append(
        Chirp(
            lr_data.intercept / sample_rate,
            lr_data.slope / sample_rate,
            chirp_len / sample_rate
        )
    )

chirps_df = pd.DataFrame.from_records(
    records,
    columns=["gap_idx_0", "gap_idx_1", "data_idx_0", "data_idx_1", "chirp_len", "period_0", "d_period", "fit_r2"]
)
chirps_df

In [ ]:
fig, ax = wide_plot()
ax.plot(data[10000:12000])
Audio(data, rate=sample_rate)

In [ ]:
sfx_0 = SoundEffect(chirps)

chirp_ys = sfx_0.synthesised(18e-3)
fig, ax = wide_plot()
ax.plot(chirp_ys[10000:12000])
Audio(chirp_ys, rate=FS_OUT)

In [ ]:
print("mean chirp length:", 1000.0 * (float(chirps_df[["chirp_len"]].mean().iloc[0]) / sample_rate), "ms")

In [ ]:
fig, ax = wide_plot()
ax.plot(data[crossings[110]:crossings[147]+30])
data[crossings[147]-5:crossings[147]+5]

In [ ]:
plt.plot(half_periods[148:206])

In [ ]:
Audio(beep_vals[idxs], rate=sample_rate)

In [ ]:
ts = np.linspace(0.0, 15.0, 800)
ys = np.tanh(np.sin(ts) * 3)
plt.plot(ts, ys)

In [ ]:
beep_crossings = np.diff((beep_vals > 0.5).astype(np.float32))
beep_crossings.shape
plt.plot(beep_crossings[211000:213700], '.')

In [ ]:
plt.plot(np.minimum(1000, np.diff(np.where(beep_crossings > 0.5)[0])))